# Join, merge, and concat

Two small tables. Run the cells from top to bottom.

`orders` has 3 rows. `payments` has 3 rows. Order 1 and 2 are in both. Order 3 is only in orders. Order 4 is only in payments.

- **merge** matches rows by a column, such as `order_id`.
- **join** matches rows by the index.
- **concat** stacks tables. It does not look for a matching id.

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3],
    "name": ["Ana", "Ben", "Cia"],
    "item": ["pen", "book", "pen"],
})

payments = pd.DataFrame({
    "order_id": [1, 2, 4],
    "name": ["Ana", "Ben", "Dan"],
    "paid": [10, 20, 40],
})

print("ORDERS")
print(orders)
print()
print("PAYMENTS")
print(payments)

ORDERS
   order_id name  item
0         1  Ana   pen
1         2  Ben  book
2         3  Cia   pen

PAYMENTS
   order_id name  paid
0         1  Ana    10
1         2  Ben    20
2         4  Dan    40


## merge

`how` decides which rows you keep.

- `inner`: only ids in both tables (1 and 2)
- `left`: every order, payment if it exists (1, 2, 3)
- `right`: every payment, order if it exists (1, 2, 4)
- `outer`: every id from either table (1, 2, 3, 4)

Both tables have a column named `name`, so pandas renames them `name_x` and `name_y`.

In [2]:
print("INNER: ids in both tables")
print(pd.merge(orders, payments, on="order_id", how="inner"))

INNER: ids in both tables
   order_id name_x  item name_y  paid
0         1    Ana   pen    Ana    10
1         2    Ben  book    Ben    20


In [3]:
print("LEFT: keep every order. Order 3 has no payment, so paid is empty.")
print(pd.merge(orders, payments, on="order_id", how="left"))

LEFT: keep every order. Order 3 has no payment, so paid is empty.
   order_id name_x  item name_y  paid
0         1    Ana   pen    Ana  10.0
1         2    Ben  book    Ben  20.0
2         3    Cia   pen    NaN   NaN


In [4]:
print("RIGHT: keep every payment. Order 4 has no order row, so item is empty.")
print(pd.merge(orders, payments, on="order_id", how="right"))

RIGHT: keep every payment. Order 4 has no order row, so item is empty.
   order_id name_x  item name_y  paid
0         1    Ana   pen    Ana    10
1         2    Ben  book    Ben    20
2         4    NaN   NaN    Dan    40


In [5]:
print("OUTER: keep every id from either table")
print(pd.merge(orders, payments, on="order_id", how="outer"))

OUTER: keep every id from either table
   order_id name_x  item name_y  paid
0         1    Ana   pen    Ana  10.0
1         2    Ben  book    Ben  20.0
2         3    Cia   pen    NaN   NaN
3         4    NaN   NaN    Dan  40.0


In [6]:
print("indicator adds a column that says where the row came from")
print(pd.merge(orders, payments, on="order_id", how="outer", indicator=True))

indicator adds a column that says where the row came from
   order_id name_x  item name_y  paid      _merge
0         1    Ana   pen    Ana  10.0        both
1         2    Ben  book    Ben  20.0        both
2         3    Cia   pen    NaN   NaN   left_only
3         4    NaN   NaN    Dan  40.0  right_only


## Same key, different column names

Use `left_on` and `right_on` when the id column is not spelled the same in both tables. The two tables below are still `orders` and `payments`. Only the id column is renamed for this example.

In [7]:
orders_renamed = orders.rename(columns={"order_id": "id_left"})
payments_renamed = payments.rename(columns={"order_id": "id_right"})

print(pd.merge(
    orders_renamed,
    payments_renamed,
    left_on="id_left",
    right_on="id_right",
    how="inner",
))

   id_left name_x  item  id_right name_y  paid
0        1    Ana   pen         1    Ana    10
1        2    Ben  book         2    Ben    20


## Two matching columns, and name clashes

`on=["order_id", "name"]` keeps a row only when both the id and the name match.

`suffixes` chooses the new names when both tables have a column that is not the key. Here that column is `name`.

In [8]:
print("Match on order_id AND name")
print(pd.merge(orders, payments, on=["order_id", "name"], how="inner"))
print()
print("Pick your own names for the shared name column")
print(pd.merge(
    orders,
    payments,
    on="order_id",
    how="left",
    suffixes=("_order", "_payment"),
))

Match on order_id AND name
   order_id name  item  paid
0         1  Ana   pen    10
1         2  Ben  book    20

Pick your own names for the shared name column
   order_id name_order  item name_payment  paid
0         1        Ana   pen          Ana  10.0
1         2        Ben  book          Ben  20.0
2         3        Cia   pen          NaN   NaN


## join

`join` uses the index, not a column name. Set `order_id` as the index first.

Both frames still have a `name` column, so `lsuffix` and `rsuffix` rename them.

In [9]:
orders_i = orders.set_index("order_id")
payments_i = payments.set_index("order_id")

print("INDEX ORDERS")
print(orders_i)
print()
print("INDEX PAYMENTS")
print(payments_i)

INDEX ORDERS
         name  item
order_id           
1         Ana   pen
2         Ben  book
3         Cia   pen

INDEX PAYMENTS
         name  paid
order_id           
1         Ana    10
2         Ben    20
4         Dan    40


In [10]:
print("join left: same rows as merge how='left'")
print(orders_i.join(payments_i, how="left", lsuffix="_order", rsuffix="_payment"))
print()
print("join inner")
print(orders_i.join(payments_i, how="inner", lsuffix="_order", rsuffix="_payment"))
print()
print("join outer")
print(orders_i.join(payments_i, how="outer", lsuffix="_order", rsuffix="_payment"))

join left: same rows as merge how='left'
         name_order  item name_payment  paid
order_id                                    
1               Ana   pen          Ana  10.0
2               Ben  book          Ben  20.0
3               Cia   pen          NaN   NaN

join inner
         name_order  item name_payment  paid
order_id                                    
1               Ana   pen          Ana    10
2               Ben  book          Ben    20

join outer
         name_order  item name_payment  paid
order_id                                    
1               Ana   pen          Ana  10.0
2               Ben  book          Ben  20.0
3               Cia   pen          NaN   NaN
4               NaN   NaN          Dan  40.0


## concat

`concat` does not match `order_id`. It pastes tables together.

- `axis=0` stacks rows (one table under the other)
- `axis=1` places tables side by side, lined up by index
- `join="outer"` keeps every column
- `join="inner"` keeps only columns that exist in every table
- `keys` adds a label so you can see which table a row came from

In [11]:
print("Stack rows. Columns that exist in only one table are empty in the other.")
print(pd.concat([orders, payments], axis=0))

Stack rows. Columns that exist in only one table are empty in the other.
   order_id name  item  paid
0         1  Ana   pen   NaN
1         2  Ben  book   NaN
2         3  Cia   pen   NaN
0         1  Ana   NaN  10.0
1         2  Ben   NaN  20.0
2         4  Dan   NaN  40.0


In [12]:
print("Same stack, but the index restarts at 0")
print(pd.concat([orders, payments], axis=0, ignore_index=True))

Same stack, but the index restarts at 0
   order_id name  item  paid
0         1  Ana   pen   NaN
1         2  Ben  book   NaN
2         3  Cia   pen   NaN
3         1  Ana   NaN  10.0
4         2  Ben   NaN  20.0
5         4  Dan   NaN  40.0


In [13]:
print("Stack rows, keep only columns that both tables have: order_id and name")
print(pd.concat([orders, payments], axis=0, join="inner"))

Stack rows, keep only columns that both tables have: order_id and name
   order_id name
0         1  Ana
1         2  Ben
2         3  Cia
0         1  Ana
1         2  Ben
2         4  Dan


In [14]:
print("Side by side by the current index 0, 1, 2. This does NOT match order_id.")
print("Row 2 of orders (Cia, id 3) sits next to row 2 of payments (Dan, id 4).")
print(pd.concat([orders, payments], axis=1))

Side by side by the current index 0, 1, 2. This does NOT match order_id.
Row 2 of orders (Cia, id 3) sits next to row 2 of payments (Dan, id 4).
   order_id name  item  order_id name  paid
0         1  Ana   pen         1  Ana    10
1         2  Ben  book         2  Ben    20
2         3  Cia   pen         4  Dan    40


In [15]:
print("Side by side after setting order_id as the index. Now row 1 sits next to row 1.")
print(pd.concat([orders_i, payments_i], axis=1, join="outer"))

Side by side after setting order_id as the index. Now row 1 sits next to row 1.
         name  item name  paid
order_id                      
1         Ana   pen  Ana  10.0
2         Ben  book  Ben  20.0
3         Cia   pen  NaN   NaN
4         NaN   NaN  Dan  40.0


In [16]:
print("keys labels each block so you can see which table it came from")
print(pd.concat([orders, payments], axis=0, keys=["orders", "payments"]))

keys labels each block so you can see which table it came from
            order_id name  item  paid
orders   0         1  Ana   pen   NaN
         1         2  Ben  book   NaN
         2         3  Cia   pen   NaN
payments 0         1  Ana   NaN  10.0
         1         2  Ben   NaN  20.0
         2         4  Dan   NaN  40.0
